# Is the resolution gap of the arrow of time uniform? — check it yourself

This notebook checks three claims of the paper from scratch in about a minute on a CPU.

1. **The gap is not uniform, and it shrinks only logarithmically.** The smallest gap $g_{\min}(\sigma) = \min(\kappa - R)/\sigma$ of the ripple strips is recomputed at $\sigma = 10^{-4}$ … $10^{-96}$ from the optimal parameters alone, with an evaluator written separately from the study code. It is compared with the fit ($\sigma \leq 10^{-24}$) $1.02/\sqrt{\ln(1/\sigma) - 23.4}$ and with two proved lower bounds of the same form.
2. **The strips form no deep valleys** (Numerical result 3): the largest eigenvalue of $-tH$ stays below $1.88$.
3. **Smoothing removes fine structure exponentially** (Lemma 5), on random structures.

The evaluator `strip_gap.py` is downloaded from the repository and its SHA-256 hash is checked before use.

In [ ]:
import hashlib, math, os, urllib.request
import numpy as np
import matplotlib.pyplot as plt

URL = 'https://raw.githubusercontent.com/neginyan/resolution-gap-log-law/main/independent_checks/strip_gap.py'
SHA256 = '9075941961c5a2ba47e02c72dcec0aa33fcdfee886423da5c65351d0f5dec353'
if not os.path.exists('strip_gap.py'):
    urllib.request.urlretrieve(URL, 'strip_gap.py')
digest = hashlib.sha256(open('strip_gap.py', 'rb').read()).hexdigest()
assert digest == SHA256, 'hash mismatch: ' + digest
import strip_gap as S
print('strip_gap.py downloaded, SHA-256 checked')

## 1. The smallest gap of the ripple strips

A ripple strip is a Gaussian state carrying a ripple of wavenumber $k_0/\sigma$ along the unstable direction, confined to a window along the stable direction. The parameters below are the optima found in the study; the gap is recomputed here.

In [ ]:
STRIPS = [  # sigma, k0, lambda, M, c, L_v, value in Table II of the paper
    (1e-4, 1.0776040913961458, 2.568927048360967, 5.843292237991376, 0.9999962543713626, 0.009826802272113236, 1.149),
    (1e-6, 1.2274723008157367, 2.1260787682706574, 19.00331868912701, 0.9999999936597486, 0.0007078256023545683, 0.740),
    (1e-12, 1.8479474435694963, 1.543676118113122, 554.9685553693851, 0.9999999970672002, 3.1779586644318827e-07, 0.335),
    (1e-24, 2.9999410048171837, 1.1723430808691142, 488764.42415849684, 0.9999999929468937, 1.4216579634831477e-13, 0.179),
    (1e-48, 4.663473586141622, 0.9316671817787303, 436034261864.8868, 0.9999999999999998, 7.145893889696438e-26, 0.110),
    (1e-96, 6.960118489440912, 0.7604602230229209, 3.883961199338749e+23, 0.9997217372476832, 3.815003067880804e-50, 0.073),
]
rows = []
for sig, k0, lam, M, c, Lv, ref in STRIPS:
    g = S.evaluate(sig, k0, lam, M, c, Lv)['gap_over_sigma']
    rows.append((sig, g)); print(f'sigma = {sig:8.0e}   g = {g:.6f}   (paper: {ref})')

Two proved lower bounds have the same form $c/\sqrt{\ln(1/\sigma) + C}$: Theorem 7 with the ratio condition $K_\Lambda \le 1$ (which holds on the strips), and Proposition 14, which needs no ratio condition and applies because the strips form no deep valleys ($\Lambda_* = 1.8745$). The prefactor $\Pi$ (close to one) is set to one.

In [ ]:
from scipy.optimize import minimize_scalar
def Lb(s): return 2 * np.log(1 / s) + np.log(2 * (np.pi ** 2 + s * s))
def g_thm7(s):
    f = lambda t: -np.sqrt(2) / (np.pi * np.sqrt((4 / t) * (2 * np.log(1 / s) + np.log(8 * (np.pi ** 2 + s * s / 2) / (np.e * t * (1 - t))))))
    return -minimize_scalar(f, bounds=(1e-3, 1 - 1e-6), method='bounded').fun
def g_prop14(s, Ls=1.8745):
    def f(tau):
        B = 8 * (1 + Ls) / (1 - tau ** (1 + Ls)); return -np.sqrt(2) / (np.pi * np.sqrt(2 + B * (np.log(1 / tau) + Lb(s))))
    return -minimize_scalar(f, bounds=(1e-6, 0.999), method='bounded').fun
ss = np.logspace(-2, -100, 200); x = np.log(1 / ss)
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot([np.log(1 / s) for s, g in rows], [g for s, g in rows], 'o', color='k', label='strips: $g_{\\min}(\\sigma)$ (recomputed here)')
xf = np.linspace(50, 230, 200); ax.plot(xf, 1.02 / np.sqrt(xf - 23.4), '-', color='grey', lw=1, label='fit ($\sigma \leq 10^{-24}$) $1.02/\\sqrt{\\ln(1/\\sigma) - 23.4}$')
ax.plot(x, [g_thm7(s) for s in ss], ':', color='#009E73', lw=1.8, label='Theorem 7 ($K_\\Lambda \\leq 1$)')
ax.plot(x, [g_prop14(s) for s in ss], '--', color='#0072B2', lw=1.5, label='Proposition 14 (no deep valleys)')
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('$\\ln(1/\\sigma)$'); ax.set_ylabel('$g = (\\kappa - R)/\\sigma$')
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

All three curves fall like $1/\sqrt{\ln(1/\sigma)}$: the gap is not uniform, but it shrinks only logarithmically, and the strips stay above the proved bounds.

## 2. The strips form no deep valleys

A deep valley is a place where the observer cannot tell from which part of the state a point came, so that the posterior spreads far beyond the blur. Here we compute the largest eigenvalue of $-tH$ for the $\sigma = 10^{-2}$ strip on a Fourier grid (the paper's value over the whole window is $1.8745$).

In [ ]:
sig, k0, lam, M, c, eta = 0.01, 0.9835318720940052, 3.501498554511305, 1.528504639428711, 0.9999999999896707, 0.32429422466066626
ell = lam * math.sqrt(sig); Lw = M * ell; Lv = eta * ell; kap = k0 / sig
v = np.arange(-7 * Lv, 7 * Lv, sig / 12); w = np.arange(-7 * Lw, 7 * Lw, sig / 2); V, W = np.meshgrid(v, w, indexing='ij')
Rh = np.fft.fft2(np.exp(-V ** 2 / (2 * Lv ** 2) - W ** 2 / (2 * Lw ** 2)) * (1 + c * np.exp(-W ** 2 / (2 * ell ** 2)) * np.cos(kap * V)))
KV, KW = np.meshgrid(2 * np.pi * np.fft.fftfreq(len(v), sig / 12), 2 * np.pi * np.fft.fftfreq(len(w), sig / 2), indexing='ij')
for m in (0.25, 0.5, 1.0):
    t = m * sig ** 2; F = Rh * np.exp(-t * (KV ** 2 + KW ** 2) / 2); d = lambda k: np.real(np.fft.ifft2(F * k))
    p = d(1.0); sv, sw = d(1j * KV) / p, d(1j * KW) / p
    Hvv = -(d(-KV ** 2) / p - sv ** 2); Hww = -(d(-KW ** 2) / p - sw ** 2); Hvw = -(d(-KV * KW) / p - sv * sw)
    tr = Hvv + Hww; lmin = tr / 2 - np.sqrt(np.maximum((tr / 2) ** 2 - (Hvv * Hww - Hvw ** 2), 0))
    print(f't = {m:4.2f} sigma^2:  largest eigenvalue of -tH = {(-t * lmin)[p > 1e-8 * p.max()].max():.4f}')

## 3. Smoothing removes fine structure exponentially (Lemma 5)

For $u = f * N(0, \sigma^2)$ and $\bar x = \sigma^2\|u'\|^2/\|u\|^2$, Lemma 5 states $\sigma^2\|u'\|^2 \le \bar x e^{-\bar x}\|f\|^2$. The ratio of the two sides must never exceed one.

In [ ]:
rng = np.random.default_rng(0); n = 4096; Lx = 200.0
xg = np.linspace(-Lx / 2, Lx / 2, n, endpoint=False); xi = 2 * np.pi * np.fft.fftfreq(n, xg[1] - xg[0]); ratios = []
for _ in range(300):
    f = sum(rng.normal() * np.exp(-(xg - rng.uniform(-20, 20)) ** 2 / (2 * rng.uniform(0.05, 3) ** 2)) * np.cos(rng.uniform(0, 8) * xg + rng.uniform(0, 6.3)) for _ in range(rng.integers(1, 6)))
    s = rng.uniform(0.1, 2.0); fh = np.fft.fft(f); uh = fh * np.exp(-s ** 2 * xi ** 2 / 2)
    xb = s ** 2 * np.sum(xi ** 2 * np.abs(uh) ** 2) / np.sum(np.abs(uh) ** 2)
    ratios.append(s ** 2 * np.sum(xi ** 2 * np.abs(uh) ** 2) / (xb * np.exp(-xb) * np.sum(np.abs(fh) ** 2)))
print(f'largest ratio over 300 random structures: {max(ratios):.6f}  (Lemma 5: never above 1)')